# BERT fine-tuning for ABSA

Train BERT (bert-base-uncased) with HuggingFace Trainer. Input: sentence [SEP] aspect.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_processed_splits
from src.preprocess import encode_labels
from src.models.bert_model import get_bert_model_and_tokenizer, prepare_bert_dataset, create_bert_trainer
from src.evaluate import compute_all_metrics, plot_confusion_matrix
from src.utils import set_seed

In [ ]:
set_seed(42)
processed_dir = ROOT / "data" / "processed"
if (processed_dir / "train.csv").exists():
    train_df, val_df, test_df = load_processed_splits(str(processed_dir))
else:
    train_df = pd.DataFrame({
        "sentence": ["Great food and service."] * 40,
        "aspect_term": ["food", "service"] * 20,
        "polarity": ["positive", "negative"] * 20,
    })
    val_df = test_df = train_df.iloc[:10]
train_df["label"] = encode_labels(train_df["polarity"])
val_df["label"] = encode_labels(val_df["polarity"])
test_df["label"] = encode_labels(test_df["polarity"])
model, tokenizer = get_bert_model_and_tokenizer("bert-base-uncased", num_labels=4)
train_ds = prepare_bert_dataset(train_df["sentence"].tolist(), train_df["aspect_term"].tolist(), train_df["label"].tolist(), tokenizer, 128)
val_ds = prepare_bert_dataset(val_df["sentence"].tolist(), val_df["aspect_term"].tolist(), val_df["label"].tolist(), tokenizer, 128)
test_ds = prepare_bert_dataset(test_df["sentence"].tolist(), test_df["aspect_term"].tolist(), test_df["label"].tolist(), tokenizer, 128)
print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))

In [ ]:
def compute_metrics(eval_pred):
    from transformers import EvalPrediction
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return compute_all_metrics(labels, preds)

trainer = create_bert_trainer(
    model=model, train_dataset=train_ds, eval_dataset=val_ds,
    output_dir=str(ROOT / "results" / "checkpoints" / "bert_notebook"),
    num_train_epochs=2, learning_rate=2e-5, batch_size=8, seed=42,
)
trainer.compute_metrics = compute_metrics
trainer.train()

In [ ]:
out = trainer.predict(test_ds)
preds = np.argmax(out.predictions, axis=-1)
metrics = compute_all_metrics(out.label_ids, preds)
print("Test metrics:", metrics)
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
plot_confusion_matrix(out.label_ids, preds, save_path=ROOT / "results" / "figures" / "notebook_bert_cm.png", title="BERT Confusion Matrix")